# Préparation du MixtureParquet Baoulé — version hackathon

Ce notebook matérialise les données sélectionnées pour le premier fine-tuning OmniASR CTC-1B. Il respecte les exclusions manuelles, diffère les audios à segmenter, conserve les validations manuelles et automatiques, et ajoute les audios Waxal non rejetés de 40 secondes maximum.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'datasets[audio]>=4.0,<5', 'huggingface_hub>=0.32,<2',
                       'numpy>=1.26,<3',
                       'pandas>=2,<3', 'pyarrow>=20,<22',
                       'scipy>=1.13,<2', 'soundfile>=0.13,<1'])
print('Dépendances de préparation installées.')

In [ ]:
from pathlib import Path
import subprocess

REPOSITORY_URL = 'https://github.com/ngangoula-jaures/Finetune_omniASR_CTC_1B_for_baoule.git'
PROJECT_DIR = Path('/kaggle/working/Finetune_omniASR_CTC_1B_for_baoule')
if not PROJECT_DIR.is_dir():
    subprocess.check_call(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_DIR)])
else:
    subprocess.check_call(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'])

PREPARE_SCRIPT = PROJECT_DIR / 'src' / 'prepare_mixture.py'
REVIEW_MANIFEST = PROJECT_DIR / 'baoule_ctc1b_review' / 'decision_manifest_reviewed.csv'
if not PREPARE_SCRIPT.is_file():
    raise FileNotFoundError('src/prepare_mixture.py absent : pousse les nouveaux fichiers vers GitHub.')
if not REVIEW_MANIFEST.is_file():
    raise FileNotFoundError('Le manifeste révisé doit être versionné dans baoule_ctc1b_review/.')
print('Projet :', PROJECT_DIR)
print('Décisions :', REVIEW_MANIFEST)

## Vérifier la sélection avant de traiter l'audio

Cette cellule ne télécharge aucun dataset. Les nombres attendus sont 1 219 exemples sélectionnés, 820 différés et 53 exclus.

In [ ]:
import json
import shutil

PREVIEW_DIR = Path('/kaggle/working/baoule_ctc1b_selection_preview')
if PREVIEW_DIR.exists():
    shutil.rmtree(PREVIEW_DIR)
subprocess.check_call([
    sys.executable, str(PREPARE_SCRIPT),
    '--review-manifest', str(REVIEW_MANIFEST),
    '--output-dir', str(PREVIEW_DIR),
    '--waxal-max-seconds', '40',
    '--selection-only',
])
preview = json.loads((PREVIEW_DIR / 'preparation_summary.json').read_text(encoding='utf-8'))
print(json.dumps(preview['totals'], ensure_ascii=False, indent=2))
assert preview['totals']['selected_examples'] == 1219
assert preview['totals']['status_counts'] == {'deferred': 820, 'excluded': 53, 'selected': 1219}

## Construire le dataset

Les audios sélectionnés sont décodés, convertis en mono 16 kHz et encodés en FLAC. Le résultat suit le schéma et le partitionnement Hive du pipeline MixtureParquet officiel de Meta. Cette cellule refuse d'écraser un dataset existant.

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/baoule_ctc1b_data')
DATASET_ROOT = OUTPUT_DIR / 'baoule_mixed' / 'version=0'
if DATASET_ROOT.exists():
    raise FileExistsError(f'{DATASET_ROOT} existe déjà. Conserve-le ou choisis un nouveau OUTPUT_DIR.')

subprocess.check_call([
    sys.executable, str(PREPARE_SCRIPT),
    '--review-manifest', str(REVIEW_MANIFEST),
    '--output-dir', str(OUTPUT_DIR),
    '--cache-dir', '/kaggle/working/hf-cache',
    '--waxal-max-seconds', '40',
    '--rows-per-file', '100',
])
summary = json.loads((OUTPUT_DIR / 'preparation_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary['materialized_groups'], ensure_ascii=False, indent=2))

## Contrôles structurels et audio

Cette vérification relit le dataset avec le partitionnement Hive, contrôle le nombre de lignes, les splits, la langue, la taille maximale et décode quelques FLAC.

In [ ]:
import io
import numpy as np
import pyarrow.dataset as pa_dataset
import soundfile as sf

dataset = pa_dataset.dataset(DATASET_ROOT, format='parquet', partitioning='hive')
required = {'text', 'audio_bytes', 'audio_size', 'corpus', 'split', 'language'}
assert required.issubset(dataset.schema.names), dataset.schema
table = dataset.to_table(columns=['text', 'audio_bytes', 'audio_size', 'corpus', 'split', 'language'])
assert table.num_rows == 1219, table.num_rows
assert set(table['language'].to_pylist()) == {'bci_Latn'}
assert set(table['split'].to_pylist()) == {'train', 'dev', 'test'}
assert max(table['audio_size'].to_pylist()) <= 40 * 16_000 + 1
assert min(table['audio_size'].to_pylist()) >= 16_000

for index in [0, table.num_rows // 2, table.num_rows - 1]:
    encoded = np.asarray(table['audio_bytes'][index].as_py(), dtype=np.int8).tobytes()
    waveform, sample_rate = sf.read(io.BytesIO(encoded), dtype='float32')
    assert sample_rate == 16_000
    assert waveform.ndim == 1
    assert len(waveform) == table['audio_size'][index].as_py()
print('MixtureParquet validé :', table.num_rows, 'exemples.')
print((OUTPUT_DIR / 'language_distribution_0.tsv').read_text(encoding='utf-8'))
print((OUTPUT_DIR / 'assets' / 'baoule_mixed.yaml').read_text(encoding='utf-8'))
assert (OUTPUT_DIR / 'README.md').is_file()

## Conserver la sortie

Après validation, utilisez **Save Version** dans Kaggle en conservant les sorties. Le dossier `/kaggle/working/baoule_ctc1b_data` servira au smoke test. Il est inutile de placer les Parquet dans GitHub.

## Publier sur Hugging Face

Dans **Add-ons → Secrets** de Kaggle, créez un secret nommé `HF_TOKEN`, collez un token Hugging Face ayant le droit d'écrire dans l'organisation `Tree-AI-lab`, puis activez son accès pour ce notebook. Le dépôt est privé par défaut. Remplacez `HF_PRIVATE = True` par `False` uniquement si vous souhaitez le publier immédiatement.

In [ ]:
import os
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

HF_REPO_ID = 'Tree-AI-lab/baoule-asr-dataset-mixture'
HF_PRIVATE = True
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Le secret Kaggle HF_TOKEN est absent ou vide.')

api = HfApi(token=HF_TOKEN)
identity = api.whoami()
print('Authentification Hugging Face vérifiée pour :', identity.get('name', 'compte inconnu'))
api.create_repo(
    repo_id=HF_REPO_ID, repo_type='dataset', private=HF_PRIVATE, exist_ok=True,
)
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'
result = api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type='dataset',
    folder_path=str(OUTPUT_DIR),
    commit_message='data: publish Baoulé hackathon MixtureParquet v0',
)
print('Publication terminée :', result)
print('Dataset :', f'https://huggingface.co/datasets/{HF_REPO_ID}')

In [ ]:
remote_files = api.list_repo_files(HF_REPO_ID, repo_type='dataset')
required_remote = {
    'README.md', 'preparation_summary.json', 'hackathon_selection_manifest.csv',
    'language_distribution_0.tsv', 'assets/baoule_mixed.yaml',
}
missing_remote = required_remote.difference(remote_files)
if missing_remote:
    raise RuntimeError(f'Fichiers absents du dépôt distant : {sorted(missing_remote)}')
parquet_files = [path for path in remote_files if path.endswith('.parquet')]
if not parquet_files:
    raise RuntimeError('Aucun fichier Parquet trouvé sur Hugging Face.')
print(f'Publication vérifiée : {len(remote_files)} fichiers, dont {len(parquet_files)} Parquet.')
del HF_TOKEN